In [1]:
import gc
from pathlib import Path

import pandas as pd
import torch
import sacrebleu
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from IPython.display import display

from src.preprocessing import preprocess_parallel_data
from src.inference import load_model
from src.generation import translate_batch

SEED = 39
BATCH_SIZE = 64
MAX_LENGTH = 50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print("Device:", DEVICE)


Device: cuda


C:\Users\festagain\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = pd.read_table(
    'data/dataset_small.tsv', quoting=3,
    names=['ru_id', 'russian', 'en_id', 'english'],
)
data, _ = preprocess_parallel_data(dataset)
train, temp = train_test_split(data, train_size=0.8, random_state=SEED, shuffle=True)
valid, test = train_test_split(temp, train_size=0.5, random_state=SEED, shuffle=True)
print(f'train: {len(train)} | valid: {len(valid)} | test: {len(test)}')

test = test[['russian', 'english']].reset_index(drop=True)
del dataset, data, train, temp, valid
gc.collect()
results = pd.DataFrame(columns=['model', 'test_size', 'BLEU'])


train: 659259 | valid: 82407 | test: 82408


In [3]:
RU_MODEL_PATH = "artifacts/tokenizers/ru_bpe.model"
EN_MODEL_PATH = "artifacts/tokenizers/en_bpe.model"

In [4]:
def evaluate_bleu(bundle, test):
    bundle['model'].to(DEVICE).eval()
    sources = test['russian'].tolist()
    references = test['english'].tolist()
    predictions = []

    for start in tqdm(range(0, len(sources), BATCH_SIZE), desc='Перевод test'):
        translated, _ = translate_batch(
            sources[start:start + BATCH_SIZE], bundle, max_length=MAX_LENGTH
        )
        predictions.extend(translated)

    score = sacrebleu.corpus_bleu(
        predictions, [references], tokenize='13a', lowercase=False,
        smooth_method='exp', use_effective_order=False,
    ).score
    del sources, references, predictions
    return float(score)


In [5]:
bundle = load_model('word_lstm', 'models/best_seq2seq_baseline.pt')
try:
    bleu = evaluate_bleu(bundle, test)

    results = results[results['model'] != 'word_lstm'].reset_index(drop=True)
    results.loc[len(results)] = ['word_lstm', len(test), bleu]
finally:
    del bundle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

display(results.round({'BLEU': 2}))


Перевод test: 100%|██████████| 1288/1288 [00:11<00:00, 114.69it/s]


,model,test_size,BLEU
0,word_lstm,82408,44.38


In [6]:
bundle = load_model(
    name="bpe_lstm",
    checkpoint_path="models/best_seq2seq_bpe_baseline.pt",
    ru_model_path=RU_MODEL_PATH,
    en_model_path=EN_MODEL_PATH
)
try:
    bleu = evaluate_bleu(bundle, test)
    results = results[results['model'] != 'bpe_lstm'].reset_index(drop=True)
    results.loc[len(results)] = ['bpe_lstm', len(test), bleu]
finally:
    del bundle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

display(results.round({'BLEU': 2}))


Перевод test: 100%|██████████| 1288/1288 [00:10<00:00, 121.85it/s]


,model,test_size,BLEU
0,word_lstm,82408,44.38
1,bpe_lstm,82408,42.79


In [7]:
bundle = load_model(
    name="bpe_attention_lstm",
    checkpoint_path="models/best_bpe_attention.pt",
    ru_model_path=RU_MODEL_PATH,
    en_model_path=EN_MODEL_PATH
)
try:
    bleu = evaluate_bleu(bundle, test)
    results = results[results['model'] != 'bpe_attention_lstm'].reset_index(drop=True)
    results.loc[len(results)] = ['bpe_attention_lstm', len(test), bleu]
finally:
    del bundle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

display(results.round({'BLEU': 2}))


Перевод test: 100%|██████████| 1288/1288 [00:18<00:00, 68.42it/s]


,model,test_size,BLEU
0,word_lstm,82408,44.38
1,bpe_lstm,82408,42.79
2,bpe_attention_lstm,82408,48.96


In [8]:
results = results.sort_values('BLEU', ascending=False).reset_index(drop=True)
display(results.round({'BLEU': 2}))
if len(results) == 3:
    print('Лучший BLEU на этом test:', results.iloc[0]['model'])
else:
    print('Выполнены не все три секции оценки.')

Path('results').mkdir(exist_ok=True)
results.to_csv('results/test_bleu_comparison.csv', index=False)


,model,test_size,BLEU
0,bpe_attention_lstm,82408,48.96
1,word_lstm,82408,44.38
2,bpe_lstm,82408,42.79


Лучший BLEU на этом test: bpe_attention_lstm
